# 02 — Candidate Generation / Blocking (Phase 4)

Measures candidate recall / candidate-count / runtime for each blocking
rule individually and for the union, using the real functions in
`src/blocking.py` and `src/retrieval.py` (the exact code path
`src/inference.py` uses at test time).

**This notebook is designed to run on the AWS SageMaker instance**
(see `aws/README_AWS.md`) — normalizing and vectorizing the full ~5M-row
Source-2/Source-3 training tables is the single heaviest step in the whole
pipeline and was intentionally *not* executed on the ~12GB-RAM development
laptop used to build this code (see `cache/smoke_test.py` for the
correctness check that *was* run locally, on a small synthetic dataset).

Run this notebook top-to-bottom on the instance; it will populate the
tables below with real numbers.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from src import blocking, config, retrieval, train as train_module
from src.data_loader import load_normalized_source
from src.labeling import candidate_recall, truth_dict_from_wide
from src.data_loader import load_ground_truth
from src.utils import timer, log_mem


In [ ]:
# Sample of TRAIN Source-1 ids for a tractable dev run (full-scale: sample_size=0)
SAMPLE_SIZE = config.EXPERIMENT_SAMPLE_SIZE
SEED = config.RANDOM_SEED

s1_norm = load_normalized_source("train", "source1")
s2_norm = load_normalized_source("train", "source2")
s3_norm = load_normalized_source("train", "source3")

sample_ids = train_module.sample_source1_ids(s1_norm["entity_id"].tolist(), SAMPLE_SIZE, SEED)
s1_sample = s1_norm[s1_norm["entity_id"].isin(set(sample_ids))].reset_index(drop=True)
truth = {k: v for k, v in truth_dict_from_wide(load_ground_truth()).items() if k in set(sample_ids)}
len(s1_sample), len(truth)


## Per-rule candidate recall (measured individually, then as a union)

In [ ]:
rows = []
for rule_fn in blocking.ALL_HASH_RULES:
    with timer(rule_fn.__name__):
        cand_s2 = rule_fn(s1_sample, s2_norm, config.MAX_BLOCK_SIZE)
        cand_s3 = rule_fn(s1_sample, s3_norm, config.MAX_BLOCK_SIZE)
    cand = pd.concat([cand_s2[["entity_id_s1", "entity_id_other"]], cand_s3[["entity_id_s1", "entity_id_other"]]])
    rec = candidate_recall(cand, truth, s1_ids=sample_ids)
    summ = blocking.summarize_candidates(cand.assign(rule=rule_fn.__name__))
    rows.append({"rule": rule_fn.__name__, **rec, **{k: v for k, v in summ.items() if k != "pairs_per_rule"}})
pd.DataFrame(rows)


## TF-IDF character n-gram retrieval recall (name + address, separately)

In [ ]:
name_vec = retrieval.fit_field_vectorizer([s1_norm["name_alnum"], s2_norm["name_alnum"], s3_norm["name_alnum"]])
addr_vec = retrieval.fit_field_vectorizer([s1_norm["address_alnum"], s2_norm["address_alnum"], s3_norm["address_alnum"]])

tfidf_rows = []
for field, vec, col in [("name", name_vec, "name_alnum"), ("address", addr_vec, "address_alnum")]:
    for k in [5, 10, 20, 30, 50]:
        with timer(f"tfidf {field} k={k}"):
            c2 = retrieval.generate_tfidf_candidates(s1_sample.rename(columns={col: col}), s2_norm, col, k=k, vectorizer=vec)
            c3 = retrieval.generate_tfidf_candidates(s1_sample.rename(columns={col: col}), s3_norm, col, k=k, vectorizer=vec)
        cand = pd.concat([c2[["entity_id_s1", "entity_id_other"]], c3[["entity_id_s1", "entity_id_other"]]])
        rec = candidate_recall(cand, truth, s1_ids=sample_ids)
        tfidf_rows.append({"field": field, "k": k, **rec, "n_candidate_pairs": len(cand.drop_duplicates())})
pd.DataFrame(tfidf_rows)


## Full union (all hash rules + both TF-IDF fields) -- the actual candidate_pairs.tsv-equivalent set

In [ ]:
from src.inference import VectorizedOtherSide, candidates_for_chunk_and_source

s2_side = VectorizedOtherSide(s2_norm, name_vec, addr_vec)
s3_side = VectorizedOtherSide(s3_norm, name_vec, addr_vec)

with timer("full union candidate generation"):
    cand_s2 = candidates_for_chunk_and_source(s1_sample, s2_side, name_vec, addr_vec, config.MAX_BLOCK_SIZE, config.TFIDF_TOP_K)
    cand_s3 = candidates_for_chunk_and_source(s1_sample, s3_side, name_vec, addr_vec, config.MAX_BLOCK_SIZE, config.TFIDF_TOP_K)

union = pd.concat([cand_s2[["entity_id_s1", "entity_id_other"]], cand_s3[["entity_id_s1", "entity_id_other"]]]).drop_duplicates()
rec = candidate_recall(union, truth, s1_ids=sample_ids)
per_s1 = union.groupby("entity_id_s1").size()
print(rec)
print(f"avg candidates/s1={per_s1.mean():.1f} median={per_s1.median():.0f} max={per_s1.max()}")


**Interpretation guide (fill in after running on AWS):** the union should
recover recall close to 1.0 at a fraction of the cost of a full Cartesian
product (~5M x ~5M pairs). If any individual rule contributes negligible
extra recall over the others, it's a candidate for removal to cut runtime;
if the union's recall ceiling is below ~0.97-0.98 the downstream classifier
cannot recover the missed true matches no matter how good it is, so that's
the first thing to improve (larger `TFIDF_TOP_K`, a rule targeting the
specific failure mode found, etc.) before touching the model at all.